In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

In [73]:
ind_arr = ['NY.GDP.MKTP.KD.ZG', 'FP.CPI.TOTL.ZG', 'SL.UEM.TOTL.ZS', 'SP.DYN.IMRT.IN', 'EG.ELC.ACCS.ZS', 'IT.NET.USER.ZS', 'NV.AGR.TOTL.ZS', 'NV.IND.TOTL.ZS', 'NE.TRD.GNFS.ZS']

In [74]:
ind_concat_url = ';'.join(ind_arr)

print(f'https://api.worldbank.org/v2/country/all/indicator/{ind_concat_url}?date=2023&source=2&downloadformat=csv')

https://api.worldbank.org/v2/country/all/indicator/NY.GDP.MKTP.KD.ZG;FP.CPI.TOTL.ZG;SL.UEM.TOTL.ZS;SP.DYN.IMRT.IN;EG.ELC.ACCS.ZS;IT.NET.USER.ZS;NV.AGR.TOTL.ZS;NV.IND.TOTL.ZS;NE.TRD.GNFS.ZS?date=2023&source=2&downloadformat=csv


In [75]:
for i in range(len(ind_arr)):
  ind_url = f'{i+1}\thttps://api.worldbank.org/v2/country/all/indicator/{ind_arr[i]}?date=2023&downloadformat=csv'
  print(ind_url, '\n')

1	https://api.worldbank.org/v2/country/all/indicator/NY.GDP.MKTP.KD.ZG?date=2023&downloadformat=csv 

2	https://api.worldbank.org/v2/country/all/indicator/FP.CPI.TOTL.ZG?date=2023&downloadformat=csv 

3	https://api.worldbank.org/v2/country/all/indicator/SL.UEM.TOTL.ZS?date=2023&downloadformat=csv 

4	https://api.worldbank.org/v2/country/all/indicator/SP.DYN.IMRT.IN?date=2023&downloadformat=csv 

5	https://api.worldbank.org/v2/country/all/indicator/EG.ELC.ACCS.ZS?date=2023&downloadformat=csv 

6	https://api.worldbank.org/v2/country/all/indicator/IT.NET.USER.ZS?date=2023&downloadformat=csv 

7	https://api.worldbank.org/v2/country/all/indicator/NV.AGR.TOTL.ZS?date=2023&downloadformat=csv 

8	https://api.worldbank.org/v2/country/all/indicator/NV.IND.TOTL.ZS?date=2023&downloadformat=csv 

9	https://api.worldbank.org/v2/country/all/indicator/NE.TRD.GNFS.ZS?date=2023&downloadformat=csv 



In [76]:
raw_df = pd.read_csv('/content/Indicator_data.csv', skiprows=4, usecols=range(5))

In [77]:
raw_df.columns

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '2023'],
      dtype='object')

In [78]:
country_codes = np.unique(raw_df['Country Code'])
indicators = np.unique(raw_df['Indicator Code'])

In [79]:
cleaned_df = raw_df.pivot(index=['Country Name', 'Country Code'], columns=['Indicator Code'], values='2023').reset_index()

In [80]:
cleaned_df

Indicator Code,Country Name,Country Code,EG.ELC.ACCS.ZS,FP.CPI.TOTL.ZG,IT.NET.USER.ZS,NE.TRD.GNFS.ZS,NV.AGR.TOTL.ZS,NV.IND.TOTL.ZS,NY.GDP.MKTP.KD.ZG,SL.UEM.TOTL.ZS,SP.DYN.IMRT.IN
0,Afghanistan,AFG,85.300000,-4.644709,15.928400,66.999578,34.743247,13.449823,2.266944,14.008000,49.900000
1,Africa Eastern and Southern,AFE,50.667516,7.399186,27.800000,56.283721,15.305933,26.467975,1.941382,7.676548,41.969085
2,Africa Western and Central,AFW,57.064737,5.221168,38.500000,NaN,24.321381,23.244644,3.644591,3.203355,57.353682
3,Albania,ALB,100.000000,4.758346,83.135592,82.352493,16.274239,23.160712,4.015417,10.669000,8.300000
4,Algeria,DZA,100.000000,9.322174,76.197601,43.739798,13.369245,37.547655,4.100000,11.645000,19.700000
...,...,...,...,...,...,...,...,...,...,...,...
261,West Bank and Gaza,PSE,100.000000,5.871102,86.637653,76.173458,4.876282,16.672838,-0.972178,NaN,16.800000
262,World,WLD,91.603476,5.733163,69.200000,58.065895,4.136826,26.193343,2.859385,4.849917,28.400000
263,"Yemen, Rep.",YEM,83.600000,NaN,NaN,NaN,NaN,NaN,NaN,16.944000,34.900000
264,Zambia,ZMB,51.100000,10.884532,16.450701,78.241887,2.230120,35.107778,5.367943,5.861000,35.000000


In [81]:
reg_url = 'https://api.worldbank.org/v2/country/?format=json&per_page=500'
response = requests.get(reg_url)
data = response.json()

In [82]:
for country in data[1]:
  iso3 = country['id']
  region = country['region']['value']

  cleaned_df.loc[cleaned_df['Country Code'] == iso3, 'Region'] = region

In [83]:
cleaned_df.columns

Index(['Country Name', 'Country Code', 'EG.ELC.ACCS.ZS', 'FP.CPI.TOTL.ZG',
       'IT.NET.USER.ZS', 'NE.TRD.GNFS.ZS', 'NV.AGR.TOTL.ZS', 'NV.IND.TOTL.ZS',
       'NY.GDP.MKTP.KD.ZG', 'SL.UEM.TOTL.ZS', 'SP.DYN.IMRT.IN', 'Region'],
      dtype='object', name='Indicator Code')

In [84]:
cleaned_df['ServiceSector'] = 100 - cleaned_df['NV.AGR.TOTL.ZS'] - cleaned_df['NV.IND.TOTL.ZS']

In [85]:
agr_condition = (cleaned_df['NV.AGR.TOTL.ZS'] > cleaned_df['ServiceSector']) & (cleaned_df['NV.AGR.TOTL.ZS'] > cleaned_df['NV.IND.TOTL.ZS'])
ind_condition = (cleaned_df['NV.IND.TOTL.ZS'] > cleaned_df['ServiceSector']) & (cleaned_df['NV.IND.TOTL.ZS'] > cleaned_df['NV.AGR.TOTL.ZS'])

In [86]:
cleaned_df['ServiceSector'].isna().sum()

np.int64(27)

In [87]:
cleaned_df.loc[cleaned_df['ServiceSector'].isna() ,'DominantSector'] = 'Missing'
cleaned_df.loc[agr_condition, 'DominantSector'] = 'Agriculture'
cleaned_df.loc[ind_condition, 'DominantSector'] = 'Industry'
cleaned_df.loc[((~agr_condition) & (~ind_condition) & (cleaned_df['ServiceSector'].isna() == False)),'DominantSector'] = 'Service'

In [88]:
hdi = pd.read_excel('/content/hdi_2023.xlsx')

In [89]:
cleaned_df.columns

Index(['Country Name', 'Country Code', 'EG.ELC.ACCS.ZS', 'FP.CPI.TOTL.ZG',
       'IT.NET.USER.ZS', 'NE.TRD.GNFS.ZS', 'NV.AGR.TOTL.ZS', 'NV.IND.TOTL.ZS',
       'NY.GDP.MKTP.KD.ZG', 'SL.UEM.TOTL.ZS', 'SP.DYN.IMRT.IN', 'Region',
       'ServiceSector', 'DominantSector'],
      dtype='object', name='Indicator Code')

In [90]:
hdi.columns

Index(['countryIsoCode', 'country', 'indexCode', 'index', 'dimension',
       'indicatorCode', 'indicator', 'year', 'yearStr', 'actualValue', 'value',
       'note'],
      dtype='object')

In [91]:
cleaned_df['HDI'] = hdi.loc[hdi['indicatorCode'] == 'hdi', 'value']

In [92]:
# Filter the hdi dataset for the 'hdi' indicator code
hdi_subset = hdi[hdi['indicatorCode'] == 'hdi']

# Create a mapping dictionary from countryIsoCode to value
hdi_map = dict(zip(hdi_subset['countryIsoCode'], hdi_subset['value']))

# Map the values to the 'HDI' column in cleaned_df using the 'Country Code'
cleaned_df['HDI'] = cleaned_df['Country Code'].map(hdi_map)

# Display the sum of null values for HDI to verify
print("Missing HDI values:", cleaned_df['HDI'].isna().sum())
cleaned_df.head()

Missing HDI values: 74


Indicator Code,Country Name,Country Code,EG.ELC.ACCS.ZS,FP.CPI.TOTL.ZG,IT.NET.USER.ZS,NE.TRD.GNFS.ZS,NV.AGR.TOTL.ZS,NV.IND.TOTL.ZS,NY.GDP.MKTP.KD.ZG,SL.UEM.TOTL.ZS,SP.DYN.IMRT.IN,Region,ServiceSector,DominantSector,HDI
0,Afghanistan,AFG,85.300000,-4.644709,15.928400,66.999578,34.743247,13.449823,2.266944,14.008000,49.900000,"Middle East, North Africa, Afghanistan & Pakistan",51.806930,Service,0.496
1,Africa Eastern and Southern,AFE,50.667516,7.399186,27.800000,56.283721,15.305933,26.467975,1.941382,7.676548,41.969085,Aggregates,58.226091,Service,NaN
2,Africa Western and Central,AFW,57.064737,5.221168,38.500000,NaN,24.321381,23.244644,3.644591,3.203355,57.353682,Aggregates,52.433975,Service,NaN
3,Albania,ALB,100.000000,4.758346,83.135592,82.352493,16.274239,23.160712,4.015417,10.669000,8.300000,Europe & Central Asia,60.565049,Service,0.810
4,Algeria,DZA,100.000000,9.322174,76.197601,43.739798,13.369245,37.547655,4.100000,11.645000,19.700000,"Middle East, North Africa, Afghanistan & Pakistan",49.083100,Service,0.763


In [96]:
cleaned_df.to_csv('cleaned_dataset.csv')

In [94]:
cleaned_df.isna().sum()

,0
Indicator Code,
Country Name,0
Country Code,0
EG.ELC.ACCS.ZS,3
FP.CPI.TOTL.ZG,41
IT.NET.USER.ZS,53
NE.TRD.GNFS.ZS,47
NV.AGR.TOTL.ZS,26
NV.IND.TOTL.ZS,25
NY.GDP.MKTP.KD.ZG,14


In [95]:
cleaned_df

Indicator Code,Country Name,Country Code,EG.ELC.ACCS.ZS,FP.CPI.TOTL.ZG,IT.NET.USER.ZS,NE.TRD.GNFS.ZS,NV.AGR.TOTL.ZS,NV.IND.TOTL.ZS,NY.GDP.MKTP.KD.ZG,SL.UEM.TOTL.ZS,SP.DYN.IMRT.IN,Region,ServiceSector,DominantSector,HDI
0,Afghanistan,AFG,85.300000,-4.644709,15.928400,66.999578,34.743247,13.449823,2.266944,14.008000,49.900000,"Middle East, North Africa, Afghanistan & Pakistan",51.806930,Service,0.496
1,Africa Eastern and Southern,AFE,50.667516,7.399186,27.800000,56.283721,15.305933,26.467975,1.941382,7.676548,41.969085,Aggregates,58.226091,Service,NaN
2,Africa Western and Central,AFW,57.064737,5.221168,38.500000,NaN,24.321381,23.244644,3.644591,3.203355,57.353682,Aggregates,52.433975,Service,NaN
3,Albania,ALB,100.000000,4.758346,83.135592,82.352493,16.274239,23.160712,4.015417,10.669000,8.300000,Europe & Central Asia,60.565049,Service,0.810
4,Algeria,DZA,100.000000,9.322174,76.197601,43.739798,13.369245,37.547655,4.100000,11.645000,19.700000,"Middle East, North Africa, Afghanistan & Pakistan",49.083100,Service,0.763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,West Bank and Gaza,PSE,100.000000,5.871102,86.637653,76.173458,4.876282,16.672838,-0.972178,NaN,16.800000,"Middle East, North Africa, Afghanistan & Pakistan",78.450880,Service,0.674
262,World,WLD,91.603476,5.733163,69.200000,58.065895,4.136826,26.193343,2.859385,4.849917,28.400000,Aggregates,69.669832,Service,NaN
263,"Yemen, Rep.",YEM,83.600000,NaN,NaN,NaN,NaN,NaN,NaN,16.944000,34.900000,"Middle East, North Africa, Afghanistan & Pakistan",NaN,Missing,0.470
264,Zambia,ZMB,51.100000,10.884532,16.450701,78.241887,2.230120,35.107778,5.367943,5.861000,35.000000,Sub-Saharan Africa,62.662102,Service,0.595
